<a href="https://colab.research.google.com/github/matpaol/MAOH/blob/main/prove/audit_completo_residui_dae_colab_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Audit completo DAE: come ottenere residui selettivi sui guasti

Questo notebook esegue automaticamente un insieme ampio ma controllato di esperimenti sul metodo di
Toma, Piltan e Kim (Sensors 2021). L'obiettivo non è alzare artificialmente il residuo, ma trovare
configurazioni che mantengano il residuo basso sui **sani mai visti** e lo aumentino sui guasti.

Vengono separati quattro obiettivi:

1. **replica forense** dei numeri `0.104 / 0.386 / 0.479`;
2. ricerca di un DAE realmente più selettivo;
3. controllo dei confondenti: regime, ampiezza, fase, registrazione e cuscinetto;
4. valutazione della CNN con e senza data leakage.

Il notebook salva un checkpoint CSV dopo ogni esperimento. Se Colab si interrompe, rieseguendo tutto
gli esperimenti già conclusi vengono saltati. È consigliata una GPU T4 o migliore.

## Come usarlo

1. Attiva la GPU in Colab.
2. Controlla `RAW_DIR` nella cella di configurazione.
3. Lascia inizialmente `MODE = 'screen'`: esegue tutte le prove a 60 epoche.
4. Esegui **Runtime > Run all**.
5. Al termine carica qui il notebook eseguito e il file `all_results.csv`.

La modalità `confirm` è prevista per una seconda esecuzione: aumenta le epoche e usa tre semi, ma
solo sulle configurazioni che avremo scelto dopo lo screening.

In [1]:
!apt-get -qq update && apt-get -qq install -y unrar
!pip -q install numpy pandas scipy matplotlib scikit-learn torch

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
import os, glob, json, math, random, subprocess, time, gc, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
from scipy.signal import hilbert
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, classification_report
from sklearn.covariance import LedoitWolf
from sklearn.svm import OneClassSVM
import torch
import torch.nn as nn
from google.colab import drive

drive.mount('/content/drive')

# ---------- CONFIGURAZIONE UTENTE ----------
RAW_DIR = '/content/drive/MyDrive/bearings_detection/raw'
OUT_DIR = '/content/drive/MyDrive/dae_complete_residual_audit'
EXTRACT_DIR = '/content/extracted_dae_complete_audit'
MODE = 'screen'             # 'screen' oppure 'confirm'
FORCE_RERUN = False         # True cancella il vantaggio dei checkpoint
RUN_CNN = True
RUN_MC_DROPOUT = True

EPOCHS = 60 if MODE == 'screen' else 200
CNN_EPOCHS = 60 if MODE == 'screen' else 150
SEEDS = [0] if MODE == 'screen' else [0, 17, 42]
TOP_CNN = 3

FS=64000; FRAME=2560; FPS=FS//FRAME
PAPER_MSE=np.array([0.104,0.386,0.479],float)
CLASS_NAMES=['normal','outer','inner']
SELU_FLOOR=-1.0507009873554805*1.6732632423543772
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LR=3e-4; DAE_BATCH=256; CNN_BATCH=64
os.makedirs(OUT_DIR,exist_ok=True); os.makedirs(EXTRACT_DIR,exist_ok=True)
RESULT_PATH=os.path.join(OUT_DIR,'all_results.csv')
print('device:',DEVICE,'| mode:',MODE,'| epochs:',EPOCHS,'| seeds:',SEEDS)

Mounted at /content/drive
device: cuda | mode: screen | epochs: 60 | seeds: [0]


## 1. Dataset e metadati

Sono mantenuti `bearing`, `recording`, `regime`, secondo e posizione del frame. Nessun test può quindi
perdere accidentalmente l'identità fisica dei dati.

In [3]:
BEARING_BY_CLASS={
  0:['K001','K002','K003','K004','K005','K006'],
  1:['KA04','KA15','KA16','KA22','KA30'],
  2:['KI04','KI14','KI16','KI17','KI18','KI21']}
CLASS_OF={b:k for k,v in BEARING_BY_CLASS.items() for b in v}; BEARINGS=list(CLASS_OF)

for b in BEARINGS:
    existing=glob.glob(os.path.join(EXTRACT_DIR,b,'**','*.mat'),recursive=True)
    archive=os.path.join(RAW_DIR,b+'.rar')
    if not existing:
        if not os.path.exists(archive): raise FileNotFoundError(archive)
        subprocess.run(['unrar','x','-o+','-inul',archive,os.path.join(EXTRACT_DIR,b)+'/'],check=True)

def load_channels(path):
    mat=loadmat(path,simplify_cells=True); root=next(k for k in mat if not k.startswith('__'))
    return {z['Name']:np.asarray(z['Data'],dtype=np.float32).ravel() for z in mat[root]['Y']}

rows=[]
for b in BEARINGS:
    for path in sorted(glob.glob(os.path.join(EXTRACT_DIR,b,'**','*.mat'),recursive=True)):
        p=os.path.basename(path).split('_'); regime='_'.join(p[:3])
        ch=load_channels(path)
        if not {'phase_current_1','phase_current_2'}.issubset(ch): continue
        rows.append({'file':path,'recording':os.path.basename(path),'regime':regime,
                     'bearing':b,'class':CLASS_OF[b],
                     'complete_seconds':min(len(ch['phase_current_1']),len(ch['phase_current_2']))//FS})
RECORDS=pd.DataFrame(rows); REGIMES=sorted(RECORDS.regime.unique())
display(RECORDS.groupby(['regime','class']).agg(recordings=('recording','size'),
        bearings=('bearing','nunique'),seconds=('complete_seconds','sum')).reset_index())
RECORDS.to_csv(os.path.join(OUT_DIR,'inventory.csv'),index=False)

,regime,class,recordings,bearings,seconds
0,N09_M07_F10,0,120,6,479
1,N09_M07_F10,1,100,5,398
2,N09_M07_F10,2,120,6,479
3,N15_M01_F10,0,120,6,480
4,N15_M01_F10,1,100,5,397
5,N15_M01_F10,2,120,6,479
6,N15_M07_F04,0,120,6,479
7,N15_M07_F04,1,100,5,397
8,N15_M07_F04,2,120,6,477
9,N15_M07_F10,0,120,6,480


In [4]:
def collect_segments(regime,channel='phase_current_1',stride=FS,max_seconds=4,
                     class_regime_override=None):
    # Una finestra dura 1 s. class_regime_override serve solo ai test forensi sbilanciati.
    xs=[]; meta=[]
    for cls in range(3):
        use_regime=(class_regime_override or {}).get(cls,regime)
        part=RECORDS[(RECORDS.regime==use_regime)&(RECORDS['class']==cls)]
        for _,r in part.iterrows():
            ch=load_channels(r.file)
            if channel=='current_diff': x=(ch['phase_current_1']-ch['phase_current_2'])/np.sqrt(2)
            elif channel=='current_mean': x=(ch['phase_current_1']+ch['phase_current_2'])/2
            elif channel=='clarke_beta': x=(ch['phase_current_1']+2*ch['phase_current_2'])/np.sqrt(3)
            else: x=ch[channel]
            stop=min(len(x),max_seconds*FS)-FS
            for start in range(0,stop+1,stride):
                s=x[start:start+FS]
                if len(s)!=FS: continue
                xs.append(s); meta.append({'class':cls,'bearing':r.bearing,'recording':r.recording,
                    'regime':use_regime,'start':start,'channel':channel})
    return np.asarray(xs,np.float32),pd.DataFrame(meta)

def frames_from_segments(segments,meta,frame_mode='fixed',rotations=1):
    # Modalità: fixed, zero_cross, peak oppure fundamental_removed.
    L=FRAME*rotations; out=[]; rows=[]
    for i,s0 in enumerate(segments):
        s=s0.astype(np.float32,copy=True); offset=0
        if frame_mode=='zero_cross':
            candidates=np.flatnonzero((s[:-1]<=0)&(s[1:]>0)); offset=int(candidates[0]) if len(candidates) else 0
        elif frame_mode=='peak': offset=int(np.argmax(s[:min(FRAME,len(s))]))
        usable=(len(s)-offset)//L
        for j in range(usable):
            z=s[offset+j*L:offset+(j+1)*L].copy()
            if frame_mode=='fundamental_removed':
                spec=np.fft.rfft(z); freqs=np.fft.rfftfreq(len(z),1/FS)
                spec[np.abs(freqs-100)<30]=0; z=np.fft.irfft(spec,n=len(z)).astype(np.float32)
            out.append(z); row=meta.iloc[i].to_dict(); row.update(segment_id=i,frame_in_segment=j)
            rows.append(row)
    return np.asarray(out,np.float32),pd.DataFrame(rows)

BASE_SEGMENTS,BASE_META=collect_segments('N15_M07_F10')
print('base:',BASE_SEGMENTS.shape,BASE_META.groupby('class').size().to_dict())

base: (1355, 64000) {0: 480, 1: 397, 2: 478}


## 2. Controlli grezzi e ricerca forense

Prima delle reti vengono misurate ampiezza e variabilità per regime, classe e cuscinetto. Il notebook
cerca anche esempi singoli vicini ai tre MSE del paper: trovarli non dimostra che siano medie di classe,
ma aiuta a capire se la Figura 8 può essere composta da esempi selezionati.

In [5]:
raw_rows=[]
for regime in REGIMES:
  for channel in ['phase_current_1','phase_current_2']:
    X,M=collect_segments(regime,channel)
    for i,x in enumerate(X):
      raw_rows.append({**M.iloc[i].to_dict(),'rms':float(np.sqrt(np.mean(x.astype(float)**2))),
          'peak':float(np.max(np.abs(x))),'below_selu':float(np.mean(x<SELU_FLOOR))})
RAW=pd.DataFrame(raw_rows); RAW.to_csv(os.path.join(OUT_DIR,'raw_audit.csv'),index=False)
display(RAW.groupby(['regime','channel','class']).agg(n=('rms','size'),rms=('rms','mean'),
    rms_sd=('rms','std'),peak=('peak','mean'),below_selu=('below_selu','mean')).reset_index())

,regime,channel,class,n,rms,rms_sd,peak,below_selu
0,N09_M07_F10,phase_current_1,0,479,1.683561,0.036192,2.875058,0.235037
1,N09_M07_F10,phase_current_1,1,398,1.683915,0.085180,2.871057,0.241004
2,N09_M07_F10,phase_current_1,2,479,1.714046,0.036268,2.909962,0.246841
3,N09_M07_F10,phase_current_2,0,479,1.695575,0.035481,2.862683,0.224682
4,N09_M07_F10,phase_current_2,1,398,1.696468,0.084151,2.962660,0.233668
5,N09_M07_F10,phase_current_2,2,479,1.725595,0.035777,2.933753,0.237789
6,N15_M01_F10,phase_current_1,0,480,0.907319,0.026987,1.822126,0.000045
7,N15_M01_F10,phase_current_1,1,397,0.867295,0.069511,1.769791,0.000043
8,N15_M01_F10,phase_current_1,2,479,0.904926,0.033050,1.821477,0.000063
9,N15_M01_F10,phase_current_2,0,480,0.920253,0.024391,1.788331,0.000046


## 3. Modello flessibile e controlli

La rete supporta:

- architettura originale, piccola e minima;
- bottleneck `32, 16, 8, 4, 2`;
- `AlphaDropout` coerente con SELU: encoder, bottleneck o tutti gli strati;
- MSE, MAE, Huber e loss tempo+spettro;
- rumore, input masking, variazioni di ampiezza e piccoli shift di fase;
- SELU letterale, SELU scalata e uscita lineare.

In [6]:
class FlexDAE(nn.Module):
    def __init__(self,input_dim=FRAME,bottleneck=32,architecture='paper',output='selu',
                 dropout=0.0,drop_position='none'):
        super().__init__()
        if architecture=='paper': enc=[input_dim,1280,640,320,128,bottleneck]; dec=[bottleneck,128,320,640,1280,input_dim]
        elif architecture=='small': enc=[input_dim,640,128,bottleneck]; dec=[bottleneck,128,640,input_dim]
        else: enc=[input_dim,320,bottleneck]; dec=[bottleneck,320,input_dim]
        self.enc=nn.ModuleList(); self.dec=nn.ModuleList(); self.drop=nn.AlphaDropout(dropout); self.p=dropout
        self.drop_position=drop_position; self.output_kind=output
        for a,b in zip(enc[:-1],enc[1:]): self.enc.append(nn.Linear(a,b))
        for a,b in zip(dec[:-1],dec[1:]): self.dec.append(nn.Linear(a,b))
        for m in list(self.enc)+list(self.dec): nn.init.normal_(m.weight,0,1/math.sqrt(m.in_features)); nn.init.zeros_(m.bias)
    def encode(self,x):
        for i,m in enumerate(self.enc):
            x=torch.selu(m(x))
            if self.p and (self.drop_position in ['all','encoder'] or (self.drop_position=='bottleneck' and i==len(self.enc)-1)): x=self.drop(x)
        return x
    def decode(self,z):
        x=z
        for i,m in enumerate(self.dec):
            x=m(x)
            last=i==len(self.dec)-1
            if not last: x=torch.selu(x)
            elif self.output_kind=='selu': x=torch.selu(x)
            if self.p and not last and self.drop_position=='all': x=self.drop(x)
        return x
    def forward(self,x): return self.decode(self.encode(x))

def choose_healthy(meta,policy='random_all',seed=0,n=2560):
    rng=np.random.default_rng(seed); h=meta.index[meta['class'].eq(0)].to_numpy()
    if policy=='one_bearing': h=meta.index[meta['class'].eq(0)&meta.bearing.eq(sorted(meta.bearing.unique())[0])].to_numpy()
    elif policy=='one_recording':
        rr=sorted(meta.loc[h,'recording'].unique())[0]; h=meta.index[meta['class'].eq(0)&meta.recording.eq(rr)].to_numpy()
    elif policy=='balanced':
        z=[]; bs=sorted(meta.loc[h,'bearing'].unique())
        for j,b in enumerate(bs):
            p=meta.index[meta['class'].eq(0)&meta.bearing.eq(b)].to_numpy(); k=n//len(bs)+(j<n%len(bs))
            z.extend(rng.choice(p,k,replace=len(p)<k));
        return np.asarray(z)
    return rng.choice(h,n,replace=len(h)<n)

def transform_fit(F,idx,kind):
    spec={'kind':kind}
    if kind=='scaled_selu': spec['scale']=1.5/np.max(np.abs(F[idx]))
    elif kind=='global_zscore': spec.update(mu=float(F[idx].mean()),sd=float(F[idx].std()))
    return spec
def transform(F,spec):
    k=spec['kind']
    if k=='scaled_selu': return F*spec['scale']
    if k=='global_zscore': return (F-spec['mu'])/spec['sd']
    if k=='frame_zscore': return (F-F.mean(1,keepdims=True))/(F.std(1,keepdims=True)+1e-8)
    return F
def residual_to_amp(r,spec):
    if spec['kind']=='scaled_selu': return r/spec['scale']
    if spec['kind']=='global_zscore': return r*spec['sd']
    return r

def corrupt(x,cfg):
    kind=cfg.get('corruption','none'); level=cfg.get('corruption_level',0.0)
    if kind=='gaussian': return x+torch.randn_like(x)*level*x.std(dim=1,keepdim=True)
    if kind=='mask': return x*(torch.rand_like(x)>level)
    if kind=='amplitude': return x*(1+(torch.rand(len(x),1,device=x.device)*2-1)*level)
    if kind=='phase':
        shifts=torch.randint(-int(level),int(level)+1,(len(x),),device=x.device)
        return torch.stack([torch.roll(v,int(s.item())) for v,s in zip(x,shifts)])
    if kind=='impulses':
        mask=(torch.rand_like(x)<.003).to(x.dtype); signs=torch.where(torch.rand_like(x)>.5,1.0,-1.0)
        return x+mask*signs*level*x.std(dim=1,keepdim=True)
    if kind=='notches':
        mask=(torch.rand_like(x)<level).to(x.dtype); return x*(1-mask)
    return x

def reconstruction_loss(pred,target,kind='mse'):
    if kind=='mae': return torch.mean(torch.abs(pred-target))
    if kind=='huber': return nn.functional.smooth_l1_loss(pred,target,beta=.1)
    base=nn.functional.mse_loss(pred,target)
    if kind=='spectral':
        a=torch.log1p(torch.abs(torch.fft.rfft(pred))); b=torch.log1p(torch.abs(torch.fft.rfft(target)))
        return base+.1*nn.functional.mse_loss(a,b)
    if kind=='envelope':
        def envelope(x):
            n=x.shape[-1]; h=torch.zeros(n,device=x.device,dtype=x.dtype); h[0]=1
            if n%2==0: h[n//2]=1; h[1:n//2]=2
            else: h[1:(n+1)//2]=2
            return torch.abs(torch.fft.ifft(torch.fft.fft(x)*h))
        return base+.1*nn.functional.mse_loss(torch.log1p(envelope(pred)),torch.log1p(envelope(target)))
    return base

## 4. Motore degli esperimenti e checkpoint

Ogni configurazione cambia una sola famiglia di fattori rispetto alla baseline. Le metriche principali sono
AUC e rapporto guasto/sano. Vengono registrati anche NMSE, MAE, quantile 95%, massimo, distanza latente
di Mahalanobis e prestazione separata per cuscinetto.

In [7]:
def cfg_id(cfg):
    return hashlib.sha1(json.dumps(cfg,sort_keys=True).encode()).hexdigest()[:12]

def score_auc(values,labels,fault):
    keep=labels!=({1:2,2:1}[fault]); return roc_auc_score((labels[keep]==fault),values[keep])

def aggregate_frame_scores(v,FM,M):
    temp=pd.DataFrame({'segment_id':FM.segment_id.to_numpy(),'v':v}).groupby('segment_id').v.mean()
    return temp.reindex(range(len(M))).to_numpy()

def train_and_evaluate(cfg,return_payload=False):
    seed=cfg.get('seed',0); np.random.seed(seed); random.seed(seed); torch.manual_seed(seed)
    override=cfg.get('class_regime_override')
    S,M=collect_segments(cfg.get('regime','N15_M07_F10'),cfg.get('channel','phase_current_1'),
                         stride=int(cfg.get('stride_fraction',1.0)*FS),class_regime_override=override)
    selection=cfg.get('segment_selection','all')
    if selection in ['first1320','random1320'] and len(M)>1320:
        if selection=='first1320': keep=np.arange(1320)
        else: keep=np.sort(np.random.default_rng(seed).choice(len(M),1320,replace=False))
        S=S[keep]; M=M.iloc[keep].reset_index(drop=True)
    elif selection=='complete_recordings':
        counts=M.groupby('recording').size(); good=set(counts[counts>=4].index)
        keep=np.flatnonzero(M.recording.isin(good)); S=S[keep]; M=M.iloc[keep].reset_index(drop=True)
    F,FM=frames_from_segments(S,M,cfg.get('frame_mode','fixed'),cfg.get('rotations',1))
    # I sani usati per addestrare il DAE possono provenire da un regime diverso da quello di valutazione.
    train_regime=cfg.get('healthy_train_regime',cfg.get('regime','N15_M07_F10'))
    if train_regime==cfg.get('regime','N15_M07_F10') and not override:
        HF,HFM=F,FM
    else:
        HS,HM=collect_segments(train_regime,cfg.get('channel','phase_current_1'))
        HF,HFM=frames_from_segments(HS,HM,cfg.get('frame_mode','fixed'),cfg.get('rotations',1))
    chosen=choose_healthy(HFM,cfg.get('healthy_policy','random_all'),seed)
    tr,va=chosen[:2048],chosen[2048:2560]; spec=transform_fit(HF,tr,cfg.get('transform','none_selu'))
    Ft=transform(F,spec).astype(np.float32)
    Ht=transform(HF,spec).astype(np.float32)
    out='selu' if cfg.get('transform','none_selu') in ['none_selu','scaled_selu'] else 'linear'
    model=FlexDAE(F.shape[1],cfg.get('bottleneck',32),cfg.get('architecture','paper'),out,
                  cfg.get('dropout',0.0),cfg.get('drop_position','none')).to(DEVICE)
    opt=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=cfg.get('weight_decay',0.0))
    T=torch.from_numpy(Ht[tr]); V=torch.from_numpy(Ht[va]).to(DEVICE)
    trajectory=[]
    checkpoints=set([1,5,10,20,40,cfg['epochs']])
    for ep in range(1,cfg['epochs']+1):
        model.train(); order=torch.randperm(len(T)); total=0
        for st in range(0,len(T),DAE_BATCH):
            target=T[order[st:st+DAE_BATCH]].to(DEVICE); inp=corrupt(target,cfg)
            z=model.encode(inp); pred=model.decode(z)
            loss=reconstruction_loss(pred,target,cfg.get('loss','mse'))
            if cfg.get('latent_l1',0): loss=loss+cfg['latent_l1']*torch.mean(torch.abs(z))
            opt.zero_grad(); loss.backward(); opt.step(); total+=loss.item()*len(target)
        if ep in checkpoints:
            model.eval()
            with torch.no_grad(): vl=float(reconstruction_loss(model(V),V,cfg.get('loss','mse')))
            point={'epoch':ep,'train_loss':total/len(T),'val_loss':vl}
            # Solo la baseline registra anche l'evoluzione dei residui: farlo per ogni ablation
            # moltiplicherebbe inutilmente il costo dello screening.
            if cfg.get('track_epochs',False):
                rng=np.random.default_rng(123); sample=[]
                for cls in range(3):
                    pool=FM.index[FM['class'].eq(cls)].to_numpy(); sample.extend(rng.choice(pool,min(700,len(pool)),replace=False))
                sample=np.asarray(sample); xs=torch.from_numpy(Ft[sample]).to(DEVICE)
                with torch.no_grad(): rr=residual_to_amp((xs-model(xs)).cpu().numpy(),spec)
                sm=np.mean(rr.astype(float)**2,axis=1); yy=FM.loc[sample,'class'].to_numpy()
                means=[sm[yy==cls].mean() for cls in range(3)]
                point.update(mse_normal=means[0],mse_outer=means[1],mse_inner=means[2],
                             ratio_outer=means[1]/means[0],ratio_inner=means[2]/means[0],
                             auc_frame_mean=(score_auc(sm,yy,1)+score_auc(sm,yy,2))/2)
            trajectory.append(point)
    model.eval(); R=np.empty_like(Ft); Z=[]
    with torch.no_grad():
        for st in range(0,len(Ft),256):
            x=torch.from_numpy(Ft[st:st+256]).to(DEVICE); pred=model(x)
            R[st:st+len(x)]=(x-pred).cpu().numpy(); Z.append(model.encode(x).cpu().numpy())
    R=residual_to_amp(R,spec); Z=np.concatenate(Z); labels=M['class'].to_numpy()
    mse_f=np.mean(R.astype(float)**2,1); mae_f=np.mean(np.abs(R.astype(float)),1)
    q95_f=np.quantile(np.abs(R.astype(float)),.95,axis=1); max_f=np.max(np.abs(R.astype(float)),1)
    energy_f=np.mean(F.astype(float)**2,1); nmse_f=mse_f/(energy_f+1e-12)
    scores={k:aggregate_frame_scores(v,FM,M) for k,v in {
        'mse':mse_f,'nmse':nmse_f,'mae':mae_f,'q95':q95_f,'max':max_f}.items()}
    # Distanza latente stimata esclusivamente sui sani realmente usati per il training.
    train_z=[]
    with torch.no_grad():
        for st in range(0,len(tr),256): train_z.append(model.encode(torch.from_numpy(Ht[tr[st:st+256]]).to(DEVICE)).cpu().numpy())
    train_z=np.concatenate(train_z); cov=LedoitWolf().fit(train_z); latent=cov.mahalanobis(Z)
    scores['latent_mahal']=aggregate_frame_scores(latent,FM,M)
    oc=OneClassSVM(kernel='rbf',gamma='scale',nu=.05).fit(train_z)
    scores['latent_ocsvm']=aggregate_frame_scores(-oc.decision_function(Z),FM,M)
    result={**cfg,'id':cfg_id(cfg),'n_segments':len(M),'n_frames':len(F),
            'below_selu_pct':100*float(np.mean(Ft<SELU_FLOOR))}
    for name,sc in scores.items():
        result[f'auc_{name}_outer']=score_auc(sc,labels,1); result[f'auc_{name}_inner']=score_auc(sc,labels,2)
        result[f'auc_{name}_mean']=(result[f'auc_{name}_outer']+result[f'auc_{name}_inner'])/2
    for cls,name in enumerate(CLASS_NAMES): result[f'mse_{name}']=float(scores['mse'][labels==cls].mean())
    result['ratio_outer']=result['mse_outer']/result['mse_normal']; result['ratio_inner']=result['mse_inner']/result['mse_normal']
    result['paper_ratio_error']=abs(np.log(result['ratio_outer']/(PAPER_MSE[1]/PAPER_MSE[0])))+abs(np.log(result['ratio_inner']/(PAPER_MSE[2]/PAPER_MSE[0])))
    per_bearing=pd.DataFrame({'bearing':M.bearing,'class':labels,'mse':scores['mse']}).groupby(['bearing','class']).mse.mean().reset_index()
    payload=(result,R.reshape(len(M),-1),M,model,spec,trajectory,per_bearing) if return_payload else result
    if not return_payload: del model,R,Z,F,Ft,Ht,S; gc.collect(); torch.cuda.empty_cache()
    return payload

def load_done():
    if FORCE_RERUN or not os.path.exists(RESULT_PATH): return pd.DataFrame()
    return pd.read_csv(RESULT_PATH)

def run_matrix(configs):
    done=load_done(); rows=[] if len(done)==0 else done.to_dict('records'); ids=set(done.id.astype(str)) if len(done) else set()
    for i,cfg in enumerate(configs,1):
        cfg={**cfg,'epochs':cfg.get('epochs',EPOCHS)}; ident=cfg_id(cfg)
        if ident in ids: print('SKIP',i,'/',len(configs),cfg['name']); continue
        print('\nRUN',i,'/',len(configs),cfg['name'])
        try: row=train_and_evaluate(cfg); rows.append(row); ids.add(ident)
        except Exception as e:
            print('ERRORE:',type(e).__name__,e); rows.append({**cfg,'id':ident,'error':repr(e)})
        pd.DataFrame(rows).to_csv(RESULT_PATH,index=False)
    return pd.DataFrame(rows)

## 5. Tutte le ablazioni sul DAE

La baseline è la replica letterale sul regime `N15_M07_F10`. Le configurazioni seguenti testano separatamente:
preprocessing, capacità, dropout, regolarizzazione, denoising, loss, segmentazione e selezione dei sani.

In [8]:
BASE=dict(regime='N15_M07_F10',channel='phase_current_1',frame_mode='fixed',rotations=1,
          stride_fraction=1.0,healthy_policy='random_all',transform='none_selu',architecture='paper',
          bottleneck=32,dropout=0.0,drop_position='none',weight_decay=0.0,corruption='none',
          corruption_level=0.0,loss='mse',seed=0)
CONFIGS=[]
def add(name,**kw): CONFIGS.append({**BASE,**kw,'name':name})

# Repliche e preprocessing
add('baseline_paper',track_epochs=True)
add('selu_scalata',transform='scaled_selu')
add('uscita_lineare',transform='linear')
add('zscore_globale',transform='global_zscore')
add('zscore_per_frame',transform='frame_zscore')
add('seconda_fase',channel='phase_current_2')
add('differenza_fasi',channel='current_diff')
add('media_fasi',channel='current_mean')
add('clarke_beta',channel='clarke_beta')

# Selezione dei sani
add('sani_bilanciati',healthy_policy='balanced')
add('sani_un_cuscinetto',healthy_policy='one_bearing')
add('sani_una_registrazione',healthy_policy='one_recording')
add('selezione_primi_1320',segment_selection='first1320')
add('selezione_casuale_1320',segment_selection='random1320')
add('solo_registrazioni_complete',segment_selection='complete_recordings')

# Capacità e bottleneck
for b in [16,8,4,2]: add(f'bottleneck_{b}',bottleneck=b)
add('architettura_piccola',architecture='small',bottleneck=16)
add('architettura_minima',architecture='tiny',bottleneck=8)

# AlphaDropout corretto per SELU
for pos in ['encoder','bottleneck','all']:
    for p in [.05,.10,.20,.30,.50]: add(f'alpha_dropout_{pos}_{p:.2f}',dropout=p,drop_position=pos)

# Weight decay
for w in [1e-6,1e-5,1e-4]: add(f'weight_decay_{w:g}',weight_decay=w)

# Denoising / spegnimento casuale degli ingressi
for p in [.05,.10,.20]: add(f'input_mask_{p:.2f}',corruption='mask',corruption_level=p)
for p in [.01,.03,.05]: add(f'rumore_gaussiano_{p:.2f}',corruption='gaussian',corruption_level=p)
for p in [.05,.10]: add(f'ampiezza_casuale_{p:.2f}',corruption='amplitude',corruption_level=p)
for p in [4,12,32]: add(f'shift_fase_{p}',corruption='phase',corruption_level=p)
for p in [1.0,2.0,4.0]: add(f'pseudo_impulsi_{p:.1f}',corruption='impulses',corruption_level=p)
for p in [.01,.03]: add(f'pseudo_buchi_{p:.2f}',corruption='notches',corruption_level=p)

# Loss
add('loss_mae',loss='mae'); add('loss_huber',loss='huber'); add('loss_tempo_spettro',loss='spectral')
add('loss_tempo_inviluppo',loss='envelope')
for lam in [1e-5,1e-4,1e-3]: add(f'codice_sparso_{lam:g}',latent_l1=lam)

# Segmentazione e rappresentazione
add('allineamento_zero',frame_mode='zero_cross')
add('allineamento_picco',frame_mode='peak')
add('fondamentale_rimossa',frame_mode='fundamental_removed',transform='linear')
add('frame_due_giri',rotations=2,architecture='small',bottleneck=16)
add('frame_quattro_giri',rotations=4,architecture='small',bottleneck=16)
add('overlap_25pct',stride_fraction=.75)
add('overlap_50pct',stride_fraction=.50)
# L'overlap 75% è escluso: quadruplica le finestre e supera la RAM di Colab.
# I test 25% e 50% sono sufficienti come controllo e hanno già mostrato risultati equivalenti.

if len(SEEDS)>1:
    CONFIGS=[{**cfg,'seed':sd,'name':f"{cfg['name']}_seed{sd}"} for cfg in CONFIGS for sd in SEEDS]
print('Esperimenti DAE:',len(CONFIGS))
ALL=run_matrix(CONFIGS)

Esperimenti DAE: 69
SKIP 1 / 69 baseline_paper
SKIP 2 / 69 selu_scalata
SKIP 3 / 69 uscita_lineare
SKIP 4 / 69 zscore_globale
SKIP 5 / 69 zscore_per_frame
SKIP 6 / 69 seconda_fase
SKIP 7 / 69 differenza_fasi
SKIP 8 / 69 media_fasi
SKIP 9 / 69 clarke_beta
SKIP 10 / 69 sani_bilanciati
SKIP 11 / 69 sani_un_cuscinetto
SKIP 12 / 69 sani_una_registrazione
SKIP 13 / 69 selezione_primi_1320
SKIP 14 / 69 selezione_casuale_1320
SKIP 15 / 69 solo_registrazioni_complete
SKIP 16 / 69 bottleneck_16
SKIP 17 / 69 bottleneck_8
SKIP 18 / 69 bottleneck_4
SKIP 19 / 69 bottleneck_2
SKIP 20 / 69 architettura_piccola
SKIP 21 / 69 architettura_minima
SKIP 22 / 69 alpha_dropout_encoder_0.05
SKIP 23 / 69 alpha_dropout_encoder_0.10
SKIP 24 / 69 alpha_dropout_encoder_0.20
SKIP 25 / 69 alpha_dropout_encoder_0.30
SKIP 26 / 69 alpha_dropout_encoder_0.50
SKIP 27 / 69 alpha_dropout_bottleneck_0.05
SKIP 28 / 69 alpha_dropout_bottleneck_0.10
SKIP 29 / 69 alpha_dropout_bottleneck_0.20
SKIP 30 / 69 alpha_dropout_bottlenec

## 6. Matrice incrociata dei regimi

Il DAE è addestrato sui sani di ciascun regime e testato separatamente sullo stesso regime. In aggiunta vengono
provate combinazioni volutamente sbilanciate fra classe e regime. Queste ultime sono test forensi: se funzionano,
dimostrano un confondente operativo, non un buon rilevatore di guasto.

In [9]:
CROSS=[]
# Matrice 4x4: regime dei sani di training x regime completo usato per la valutazione.
for train_regime in REGIMES:
    for test_regime in REGIMES:
        for channel in ['phase_current_1','phase_current_2']:
            CROSS.append({**BASE,
                'name':f'cross_train_{train_regime}_test_{test_regime}_{channel[-1]}',
                'healthy_train_regime':train_regime,'regime':test_regime,'channel':channel})

# Guasti a basso carico e sani ad alto carico, poi configurazione inversa.
CROSS += [
 {**BASE,'name':'forense_sano_alto_guasti_bassi',
  'class_regime_override':{0:'N15_M07_F10',1:'N15_M01_F10',2:'N15_M01_F10'}},
 {**BASE,'name':'forense_sano_basso_guasti_alti','regime':'N15_M01_F10',
  'class_regime_override':{0:'N15_M01_F10',1:'N15_M07_F10',2:'N15_M07_F10'}},
 {**BASE,'name':'forense_tre_regimi',
 'class_regime_override':{0:'N15_M07_F10',1:'N15_M01_F10',2:'N09_M07_F10'}},
]
if len(SEEDS)>1:
    CROSS=[{**cfg,'seed':sd,'name':f"{cfg['name']}_seed{sd}"} for cfg in CROSS for sd in SEEDS]
ALL=run_matrix(CONFIGS+CROSS)

SKIP 1 / 104 baseline_paper
SKIP 2 / 104 selu_scalata
SKIP 3 / 104 uscita_lineare
SKIP 4 / 104 zscore_globale
SKIP 5 / 104 zscore_per_frame
SKIP 6 / 104 seconda_fase
SKIP 7 / 104 differenza_fasi
SKIP 8 / 104 media_fasi
SKIP 9 / 104 clarke_beta
SKIP 10 / 104 sani_bilanciati
SKIP 11 / 104 sani_un_cuscinetto
SKIP 12 / 104 sani_una_registrazione
SKIP 13 / 104 selezione_primi_1320
SKIP 14 / 104 selezione_casuale_1320
SKIP 15 / 104 solo_registrazioni_complete
SKIP 16 / 104 bottleneck_16
SKIP 17 / 104 bottleneck_8
SKIP 18 / 104 bottleneck_4
SKIP 19 / 104 bottleneck_2
SKIP 20 / 104 architettura_piccola
SKIP 21 / 104 architettura_minima
SKIP 22 / 104 alpha_dropout_encoder_0.05
SKIP 23 / 104 alpha_dropout_encoder_0.10
SKIP 24 / 104 alpha_dropout_encoder_0.20
SKIP 25 / 104 alpha_dropout_encoder_0.30
SKIP 26 / 104 alpha_dropout_encoder_0.50
SKIP 27 / 104 alpha_dropout_bottleneck_0.05
SKIP 28 / 104 alpha_dropout_bottleneck_0.10
SKIP 29 / 104 alpha_dropout_bottleneck_0.20
SKIP 30 / 104 alpha_dropout

## 7. Graduatorie e diagnosi per cuscinetto

Vengono create due classifiche diverse:

- migliore separazione reale secondo la massima AUC disponibile;
- maggiore somiglianza ai rapporti MSE del paper.

La seconda classifica è forense e non va confusa con la bontà del modello.

In [10]:
OK=ALL[ALL.get('error',pd.Series(index=ALL.index,dtype=object)).isna()].copy()
auc_cols=[x for x in OK.columns if x.startswith('auc_') and x.endswith('_mean') and 'latent' not in x]
OK['best_auc']=OK[auc_cols].max(axis=1); OK['best_score_kind']=OK[auc_cols].idxmax(axis=1)
if 'auc_latent_mahal_mean' in OK: OK['latent_auc']=OK['auc_latent_mahal_mean']
cols=['name','mse_normal','mse_outer','mse_inner','ratio_outer','ratio_inner','best_auc','best_score_kind','paper_ratio_error']
print('Migliori per separazione:'); display(OK.sort_values('best_auc',ascending=False)[cols].head(15))
print('Più vicini ai rapporti del paper:'); display(OK.sort_values('paper_ratio_error')[cols].head(15))
OK.sort_values('best_auc',ascending=False).to_csv(os.path.join(OUT_DIR,'ranking_by_auc.csv'),index=False)
OK.sort_values('paper_ratio_error').to_csv(os.path.join(OUT_DIR,'ranking_paper_forensic.csv'),index=False)

# Ricalcola i primi candidati e include sempre la baseline per l'analisi forense.
TOP_NAMES=OK.sort_values('best_auc',ascending=False).name.head(TOP_CNN).tolist()
baseline_names=[n for n in OK.name if str(n).startswith('baseline_paper')]
if baseline_names and baseline_names[0] not in TOP_NAMES: TOP_NAMES.append(baseline_names[0])
PAYLOADS={}
all_cfg={x['name']:x for x in CONFIGS+CROSS}
for name in TOP_NAMES:
    cfg={**all_cfg[name],'epochs':EPOCHS}; print('Payload:',name)
    PAYLOADS[name]=train_and_evaluate(cfg,return_payload=True)
    PAYLOADS[name][-1].to_csv(os.path.join(OUT_DIR,f'per_bearing_{name}.csv'),index=False)
    pd.DataFrame(PAYLOADS[name][-2]).to_csv(os.path.join(OUT_DIR,f'trajectory_{name}.csv'),index=False)

# Per la baseline: distribuzione per registrazione e segmenti più vicini ai tre numeri del paper.
if baseline_names:
    bname=baseline_names[0]; _,BR,BM,*_=PAYLOADS[bname]
    bmse=np.mean(BR.astype(float)**2,axis=1)
    forensic=pd.DataFrame({'recording':BM.recording,'bearing':BM.bearing,'regime':BM.regime,
                           'class':BM['class'],'segment_mse':bmse})
    forensic.groupby(['recording','bearing','regime','class']).segment_mse.agg(['mean','median','min','max']).reset_index().to_csv(
        os.path.join(OUT_DIR,'baseline_per_recording.csv'),index=False)
    nearest=[]
    for cls,target in enumerate(PAPER_MSE):
        part=forensic[forensic['class']==cls].copy(); part['distance']=abs(part.segment_mse-target)
        nearest.append(part.nsmallest(20,'distance').assign(paper_target=target))
    pd.concat(nearest).to_csv(os.path.join(OUT_DIR,'segments_nearest_paper_mse.csv'),index=False)

Migliori per separazione:


,name,mse_normal,mse_outer,mse_inner,ratio_outer,ratio_inner,best_auc,best_score_kind,paper_ratio_error
103,forense_tre_regimi,0.082270,0.553521,4.695366,6.728087,57.072484,1.000000,auc_mse_mean,3.111857
101,forense_sano_alto_guasti_bassi,0.082270,0.553521,0.491447,6.728087,5.973565,1.000000,auc_mse_mean,0.854879
102,forense_sano_basso_guasti_alti,0.005984,0.598805,0.644698,100.067921,107.737191,1.000000,auc_mse_mean,6.446788
69,cross_train_N09_M07_F10_test_N09_M07_F10_1,0.061708,0.084595,0.078910,1.370900,1.278770,0.775755,auc_nmse_mean,2.277390
3,zscore_globale,0.009025,0.011599,0.011304,1.285172,1.252411,0.771955,auc_nmse_mean,2.362793
79,cross_train_N15_M01_F10_test_N15_M01_F10_1,0.005984,0.010754,0.007078,1.797060,1.182820,0.737336,auc_nmse_mean,2.084703
2,uscita_lineare,0.007661,0.009911,0.009690,1.293589,1.264801,0.730872,auc_nmse_mean,2.346421
62,allineamento_zero,0.079790,0.083660,0.092201,1.048498,1.155549,0.716961,auc_mae_mean,2.646821
80,cross_train_N15_M01_F10_test_N15_M01_F10_2,0.007061,0.011044,0.007551,1.564037,1.069370,0.702626,auc_max_mean,2.324416
10,sani_un_cuscinetto,0.084400,0.090427,0.097377,1.071411,1.153765,0.701904,auc_mae_mean,2.626749


Più vicini ai rapporti del paper:


,name,mse_normal,mse_outer,mse_inner,ratio_outer,ratio_inner,best_auc,best_score_kind,paper_ratio_error
101,forense_sano_alto_guasti_bassi,0.082270,0.553521,0.491447,6.728087,5.973565,1.000000,auc_mse_mean,0.854879
79,cross_train_N15_M01_F10_test_N15_M01_F10_1,0.005984,0.010754,0.007078,1.797060,1.182820,0.737336,auc_nmse_mean,2.084703
69,cross_train_N09_M07_F10_test_N09_M07_F10_1,0.061708,0.084595,0.078910,1.370900,1.278770,0.775755,auc_nmse_mean,2.277390
70,cross_train_N09_M07_F10_test_N09_M07_F10_2,0.057948,0.081968,0.070536,1.414523,1.217228,0.679875,auc_max_mean,2.295387
80,cross_train_N15_M01_F10_test_N15_M01_F10_2,0.007061,0.011044,0.007551,1.564037,1.069370,0.702626,auc_max_mean,2.324416
2,uscita_lineare,0.007661,0.009911,0.009690,1.293589,1.264801,0.730872,auc_nmse_mean,2.346421
3,zscore_globale,0.009025,0.011599,0.011304,1.285172,1.252411,0.771955,auc_nmse_mean,2.362793
8,clarke_beta,0.080013,0.096839,0.097599,1.210297,1.219787,0.693851,auc_max_mean,2.449214
1,selu_scalata,0.013299,0.016181,0.015843,1.216764,1.191330,0.700412,auc_nmse_mean,2.467491
5,seconda_fase,0.076934,0.087007,0.089265,1.130930,1.160284,0.669860,auc_max_mean,2.567051


Payload: forense_tre_regimi
Payload: forense_sano_alto_guasti_bassi
Payload: forense_sano_basso_guasti_alti
Payload: baseline_paper


## 8. Monte Carlo AlphaDropout

Per il miglior modello con dropout, il dropout viene mantenuto attivo al test e ogni frame viene ricostruito
20 volte. La varianza delle ricostruzioni misura l'incertezza del DAE. È distinta dall'MSE tradizionale.

In [11]:
if RUN_MC_DROPOUT:
    drop_candidates=OK[OK['dropout'].fillna(0)>0].sort_values('best_auc',ascending=False)
    if len(drop_candidates):
        name=drop_candidates.iloc[0]['name']; cfg={**all_cfg[name],'epochs':EPOCHS}
        result,R,M,model,spec,traj,pb=train_and_evaluate(cfg,return_payload=True)
        S,_=collect_segments(cfg['regime'],cfg['channel']); F,FM=frames_from_segments(S,M,cfg['frame_mode'],cfg['rotations'])
        Ft=transform(F,spec).astype(np.float32); model.train()
        # Calcolo streaming di media e varianza (algoritmo di Welford). La vecchia versione
        # conservava 20 ricostruzioni complete e poteva superare 6 GB di RAM.
        uncertainty=np.empty(len(Ft),dtype=np.float32)
        with torch.no_grad():
            for st in range(0,len(Ft),128):
                x=torch.from_numpy(Ft[st:st+128]).to(DEVICE)
                mean=torch.zeros_like(x); m2=torch.zeros_like(x)
                for rep in range(1,21):
                    pred=model(x); delta=pred-mean; mean=mean+delta/rep; m2=m2+delta*(pred-mean)
                uncertainty[st:st+len(x)]=(m2/19).mean(1).cpu().numpy()
        unc_seg=aggregate_frame_scores(uncertainty,FM,M); y=M['class'].to_numpy()
        mc=pd.DataFrame([{'name':name,'auc_uncertainty_outer':score_auc(unc_seg,y,1),
                         'auc_uncertainty_inner':score_auc(unc_seg,y,2)}])
        display(mc); mc.to_csv(os.path.join(OUT_DIR,'mc_dropout.csv'),index=False)
        del R,model,F,Ft,S,uncertainty; gc.collect(); torch.cuda.empty_cache()

,name,auc_uncertainty_outer,auc_uncertainty_inner
0,alpha_dropout_bottleneck_0.05,0.534446,0.556437


## 9. CNN e tre livelli di leakage

La CNN è addestrata sui residui dei candidati migliori con split casuale per segmento, per registrazione e per
cuscinetto. La stessa architettura e gli stessi residui sono mantenuti fissi.

In [12]:
def group_split(M,level,seed=0,test_fraction=.2):
    rng=np.random.default_rng(seed); tr=[];te=[]
    for cls in range(3):
        part=M[M['class']==cls]
        if level=='random_segment': groups=np.array(part.index); key=None
        else: groups=np.array(sorted(part[level].unique())); key=level
        rng.shuffle(groups); n=max(1,int(round(len(groups)*test_fraction))); test=set(groups[:n])
        if key is None: te+=list(test); tr+=list(set(part.index)-test)
        else: te+=part.index[part[key].isin(test)].tolist(); tr+=part.index[~part[key].isin(test)].tolist()
    return np.asarray(tr),np.asarray(te)

class PaperCNN(nn.Module):
    def __init__(self,L):
        super().__init__(); self.c1=nn.Conv1d(1,64,3); self.c2=nn.Conv1d(64,32,3); self.p=nn.MaxPool1d(2)
        n1=(L-2)//2; n2=(n1-2)//2; self.fc=nn.Linear(n2*32,3)
    def forward(self,x):
        x=self.p(torch.relu(self.c1(x[:,None,:]))); x=self.p(torch.relu(self.c2(x))); return self.fc(x.flatten(1))

def cnn_test(X,M,level,seed=0):
    tr,te=group_split(M,level,seed); torch.manual_seed(seed); model=PaperCNN(X.shape[1]).to(DEVICE)
    opt=torch.optim.Adam(model.parameters(),lr=LR); lossfn=nn.CrossEntropyLoss(); T=torch.from_numpy(X.astype(np.float32)); y=torch.from_numpy(M['class'].to_numpy(np.int64))
    for ep in range(CNN_EPOCHS):
        model.train(); order=torch.randperm(len(tr))
        for st in range(0,len(tr),CNN_BATCH):
            idx=tr[order[st:st+CNN_BATCH]]; xb=T[idx].to(DEVICE); yb=y[idx].to(DEVICE)
            loss=lossfn(model(xb),yb); opt.zero_grad(); loss.backward(); opt.step()
    model.eval(); pred=[]
    with torch.no_grad():
        for st in range(0,len(te),CNN_BATCH): pred.extend(model(T[te[st:st+CNN_BATCH]].to(DEVICE)).argmax(1).cpu().numpy())
    truth=y[te].numpy(); return {'split':level,'accuracy':accuracy_score(truth,pred),'n_train':len(tr),'n_test':len(te),
        'bearing_overlap':len(set(M.loc[tr,'bearing'])&set(M.loc[te,'bearing'])),
        'recording_overlap':len(set(M.loc[tr,'recording'])&set(M.loc[te,'recording']))}

CNN_ROWS=[]
if RUN_CNN:
  for name,payload in PAYLOADS.items():
    _,R,M,*_=payload
    for level in ['random_segment','recording','bearing']:
      print('CNN',name,level); CNN_ROWS.append({'name':name,**cnn_test(R,M,level)})
      pd.DataFrame(CNN_ROWS).to_csv(os.path.join(OUT_DIR,'cnn_results.csv'),index=False)
display(pd.DataFrame(CNN_ROWS))

CNN forense_tre_regimi random_segment
CNN forense_tre_regimi recording
CNN forense_tre_regimi bearing
CNN forense_sano_alto_guasti_bassi random_segment
CNN forense_sano_alto_guasti_bassi recording
CNN forense_sano_alto_guasti_bassi bearing
CNN forense_sano_basso_guasti_alti random_segment
CNN forense_sano_basso_guasti_alti recording
CNN forense_sano_basso_guasti_alti bearing
CNN baseline_paper random_segment
CNN baseline_paper recording
CNN baseline_paper bearing


,name,split,accuracy,n_train,n_test,bearing_overlap,recording_overlap
0,forense_tre_regimi,random_segment,1.000000,1085,271,17,201
1,forense_tre_regimi,recording,1.000000,1084,272,17,0
2,forense_tre_regimi,bearing,1.000000,1116,240,0,0
3,forense_sano_alto_guasti_bassi,random_segment,0.797048,1085,271,17,200
4,forense_sano_alto_guasti_bassi,recording,0.808824,1084,272,17,0
5,forense_sano_alto_guasti_bassi,bearing,0.695833,1116,240,0,0
6,forense_sano_basso_guasti_alti,random_segment,0.715867,1084,271,17,202
7,forense_sano_basso_guasti_alti,recording,0.686347,1084,271,17,0
8,forense_sano_basso_guasti_alti,bearing,0.764706,1117,238,0,0
9,baseline_paper,random_segment,0.476015,1084,271,17,202


## 10. Riepilogo finale

Il notebook termina creando un manifesto e un archivio ZIP con CSV e configurazioni. I modelli non vengono
inseriti nello ZIP per mantenerlo leggero. Carica qui il notebook eseguito e lo ZIP.

In [13]:
summary={
 'mode':MODE,'epochs':EPOCHS,'seeds':SEEDS,'n_experiments':int(len(OK)),
 'top_by_auc':OK.sort_values('best_auc',ascending=False)[['name','best_auc','best_score_kind']].head(10).to_dict('records'),
 'top_paper_forensic':OK.sort_values('paper_ratio_error')[['name','paper_ratio_error','ratio_outer','ratio_inner']].head(10).to_dict('records')}
with open(os.path.join(OUT_DIR,'summary.json'),'w') as f: json.dump(summary,f,indent=2)
archive='/content/dae_complete_residual_audit_results.zip'
!cd "$OUT_DIR" && zip -q -r "$archive" . -x '*.pt'
print('FINITO. Risultati:',OUT_DIR)
print('Archivio scaricabile:',archive)
display(OK.sort_values('best_auc',ascending=False)[['name','best_auc','best_score_kind','ratio_outer','ratio_inner']].head(10))

FINITO. Risultati: /content/drive/MyDrive/dae_complete_residual_audit
Archivio scaricabile: /content/dae_complete_residual_audit_results.zip


,name,best_auc,best_score_kind,ratio_outer,ratio_inner
103,forense_tre_regimi,1.000000,auc_mse_mean,6.728087,57.072484
101,forense_sano_alto_guasti_bassi,1.000000,auc_mse_mean,6.728087,5.973565
102,forense_sano_basso_guasti_alti,1.000000,auc_mse_mean,100.067921,107.737191
69,cross_train_N09_M07_F10_test_N09_M07_F10_1,0.775755,auc_nmse_mean,1.370900,1.278770
3,zscore_globale,0.771955,auc_nmse_mean,1.285172,1.252411
79,cross_train_N15_M01_F10_test_N15_M01_F10_1,0.737336,auc_nmse_mean,1.797060,1.182820
2,uscita_lineare,0.730872,auc_nmse_mean,1.293589,1.264801
62,allineamento_zero,0.716961,auc_mae_mean,1.048498,1.155549
80,cross_train_N15_M01_F10_test_N15_M01_F10_2,0.702626,auc_max_mean,1.564037,1.069370
10,sani_un_cuscinetto,0.701904,auc_mae_mean,1.071411,1.153765
